In [1]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain.embeddings import HuggingFaceEmbeddings
from tqdm import tqdm 
import pandas as pd
from rich import print

In [80]:
df = pd.read_csv("/Users/justinvhuang/Desktop/ISYE-CSE-MGT-6748-Group-1/data/unclassified_postings_v2.csv", index_col = 0)

In [ ]:
def count_words(text):
    words = text.split()
    return len(words)
df['word_count'] = df['body'].apply(count_words)

In [67]:
model_name = "nomic-ai/nomic-embed-text-v1.5"
model_kwargs = {'device': 'cpu', 'trust_remote_code': True}
encode_kwargs = {'normalize_embeddings': True} 
embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs,
)

/Users/justinvhuang/miniconda3/envs/dspy/lib/python3.11/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
<All keys matched successfully>


In [ ]:
text_splitter = SemanticChunker(embeddings=embeddings,
                                 breakpoint_threshold_type="percentile",
                                 breakpoint_threshold_amount=25,
                                 )

In [68]:
df2 = df.sample(1000, random_state = 10)

In [69]:
texts = df2['body'].tolist()

In [71]:
sentence_list = [None] * len(texts) 
for i, text in enumerate(tqdm(texts)):
    chunks = text_splitter.create_documents([text])
    split_text = [chunk.page_content for chunk in chunks]
    sentence_list[i] = split_text

100%|██████████| 1000/1000 [1:26:59<00:00,  5.22s/it]   


In [74]:
df2['sentence_split'] = sentence_list

In [76]:
df2[['body', 'sentence_split']]

,body,sentence_split
193211,"Software Developer\nFinance, Investment Bankin...","[Software Developer\nFinance, Investment Banki..."
93011,"['LAW OFFICE OF THEODORE MALONEY', 'Associate ...","[['LAW OFFICE OF THEODORE MALONEY', 'Associate..."
18559,Receptionist - Dental Prectice 2351 South Arli...,[Receptionist - Dental Prectice 2351 South Arl...
214593,Solar Apprentice Radiance / Sunshine Solar - A...,[Solar Apprentice Radiance / Sunshine Solar - ...
531,Fund Accountant Confidential Priviately-Held C...,[Fund Accountant Confidential Priviately-Held ...
...,...,...
31476,un Private Eye Type Task (Rock Springs) compen...,[un Private Eye Type Task (Rock Springs) compe...
105632,"SC INTERNATIONAL LTD Posted Under: Seattle, Wa...","[SC INTERNATIONAL LTD Posted Under: Seattle, W..."
142633,Communications/Special Events Coordinator\nCit...,[Communications/Special Events Coordinator\nCi...
19505,Dental Front Office - 87953 20301 Pleasant Pla...,[Dental Front Office - 87953 20301 Pleasant Pl...


In [78]:
df2.to_csv('/Users/justinvhuang/Desktop/ISYE-CSE-MGT-6748-Group-1/data/sample_split_posting.csv')